<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn · <span style='color:#3777FF'>Retention Management</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 02 — Modelos base, rede neuronal e ensembles · avaliação contra o benchmark IBM · priorização por valor</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice — Estrutura do notebook</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Implementa os <b style='color:#3777FF'>próximos passos</b> definidos na secção 14 do notebook 01.
Pré-requisito: correr primeiro <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>01_preprocessing_eda.ipynb</code> ou <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_pipeline.py</code> (gera <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code>).</p>
<p>Documentação de apoio: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/models.md</code> (como funciona cada modelo) e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/modelling.md</code> (protocolo, métricas e decisões). O mesmo fluxo corre sem notebook com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>python scripts/run_modelling.py</code>.</p>
<ol style='margin:4px 0'>
<li>Setup do ambiente</li><li>Carregamento dos artefactos</li><li>Os modelos em avaliação</li>
<li>Tuning dos modelos base</li><li>Comparação em validação cruzada e ensembles</li>
<li>Satisfaction Score — com e sem</li><li>City com TargetEncoder</li>
<li>Avaliação final no holdout</li><li>Interpretação</li><li>Priorização P(churn) × CLTV</li>
<li>Exportação</li><li>Conclusões para o relatório</li></ol></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup do ambiente</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Mesmo setup do notebook 01. Dois módulos do projeto fazem o trabalho desta fase:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling</code>: modelos, tuning, ensembles, métricas, avaliação e priorização</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>model_plots</code>: gráficos da modelação, com a paleta do projeto</li></ul>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>N_ITER</code> controla o nº de combinações testadas por modelo no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code>. Com 20, a secção 4 demora alguns minutos; baixar para 5 para testar depressa.</p></div></div>

In [ ]:
# --- Bibliotecas ---
import sys
import warnings
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.exceptions import ConvergenceWarning

# --- Encontrar a pasta do projeto (a que tem a pasta "src") ---
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# --- Módulos do próprio projeto (pasta src/telco_churn) ---
from telco_churn import model_plots, modelling, plots
from telco_churn.config import ARTIFACTS_DIR, DATA_PROCESSED, TAB_DIR, ensure_dirs
from telco_churn.style import apply_theme, style_table

# --- Opções ---
warnings.filterwarnings("ignore")         # esconder avisos
pd.set_option("display.max_columns", 60)  # mostrar mais colunas nas tabelas

SHOW = True    # True = mostrar os gráficos (são sempre guardados em ficheiro)
N_ITER = 20    # nº de combinações testadas por modelo (menos = mais rápido)

ensure_dirs()
apply_theme()
print("Pasta do projeto:", ROOT)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Carregamento dos artefactos</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Carrega <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code> (listas de features e target) e os CSV de treino, teste e Joined.
Os CSV são lidos com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>keep_default_na=False</code> para preservar o literal <i>None</i> em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Offer</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Internet Type</code>.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> não foi exportada nos CSV de treino/teste (alta cardinalidade); é recuperada da base unificada por <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Customer ID</code> só para o teste da secção 7.</p></div></div>

In [ ]:
# Carregar os dados preparados no notebook 01
train, test, joined, cfg = modelling.load_processed()

NUM = cfg["features_num"]   # colunas numéricas
CAT = cfg["features_cat"]   # colunas categóricas
TARGET = cfg["target"]      # coluna que queremos prever (churn)
FEATURES = NUM + CAT        # todas as colunas usadas pelo modelo

# X = colunas de entrada, y = resposta (churn)
X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("Treino:", len(train), "clientes | taxa de churn:", round(y_train.mean(), 3))
print("Teste: ", len(test), "clientes | taxa de churn:", round(y_test.mean(), 3))
print("Joined:", len(joined), "clientes (sem resposta, só para prever)")
print(len(NUM), "numéricas +", len(CAT), "categóricas =", len(FEATURES), "features")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Os modelos em avaliação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Comparamos <b style='color:#3777FF'>sete modelos base</b>, uma <b style='color:#3777FF'>linha de base</b> e <b style='color:#0EE071'>dois ensembles</b>. Os modelos base cobrem as grandes famílias de classificadores:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Linear</b>: Logistic Regression, a referência simples e interpretável</li><li><b style='color:#3777FF'>Árvore</b> e <b style='color:#3777FF'>distância</b>: Decision Tree e k-Nearest Neighbors, os modelos das aulas 6 e 7</li><li><b style='color:#3777FF'>Rede neuronal</b>: Neural Network (MLP), que generaliza a regressão logística com camadas escondidas</li><li><b style='color:#3777FF'>Ensembles de árvores</b>: Random Forest e Extra Trees (bagging) e Gradient Boosting (boosting)</li></ul>
<p>A tabela mostra como funciona cada um e que pré-processador usa: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>linear</code> (com escala, para modelos que somam variáveis ou medem distâncias) ou <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>trees</code> (sem escala). A explicação completa, com os hiperparâmetros, está em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>docs/models.md</code>.</p></div></div>

In [ ]:
# Família, pré-processador e funcionamento de cada modelo
display(style_table(modelling.MODEL_INFO))
print("Membros dos ensembles:", ", ".join(modelling.ENSEMBLE_MEMBERS))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>3.1 · A rede neuronal (MLP)</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um <b style='color:#3777FF'>multilayer perceptron</b> recebe as variáveis escaladas, passa-as por uma ou duas <b style='color:#3777FF'>camadas escondidas</b> de neurónios com ativação não linear (ReLU ou tanh) e termina num neurónio com <b style='color:#3777FF'>sigmoide</b>, que devolve P(churn). Os pesos são aprendidos por backpropagation (otimizador Adam), minimizando o log loss.</p>
<p>A ligação à regressão: uma <b style='color:#0EE071'>regressão logística é uma rede neuronal sem camadas escondidas</b>, um único neurónio sigmoide. As camadas escondidas permitem à rede criar as suas próprias combinações das variáveis. Uma rede de <i>regressão</i> (saída contínua, perda quadrática) não serve aqui, porque o alvo é binário: usamos a mesma arquitetura com saída sigmoide e log loss.</p>
<p>O tuning escolhe o nº de camadas e neurónios, a ativação, a regularização L2 (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>alpha</code>) e a taxa de aprendizagem. O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>early_stopping</code> separa 15% do treino e pára quando a accuracy nessa validação interna deixa de melhorar.</p></div></div>

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>3.2 · As nove métricas</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Cada modelo é avaliado com as mesmas nove métricas do notebook das aulas. As quatro primeiras usam as probabilidades; as cinco últimas precisam de uma decisão sim/não e usam o limiar 0,5.
A <b style='color:#3777FF'>ROC AUC</b> é a métrica de seleção, porque a equipa de retenção trabalha com uma lista ordenada; o <b style='color:#3777FF'>Brier</b> é o critério secundário, porque P(churn) multiplica o CLTV na priorização.</p></div></div>

In [ ]:
display(style_table(modelling.metrics_glossary()))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · Tuning dos modelos base</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Cada modelo é um <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Pipeline([("prep", ...), ("clf", ...)])</code>: o pré-processador é ajustado <b style='color:#0EE071'>dentro de cada fold</b>, por isso não há fuga de informação na validação cruzada.</p>
<p>Procura aleatória (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code>) com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StratifiedKFold</code> (5 folds) <b style='color:#0EE071'>só no treino</b>, métrica ROC AUC, para os sete modelos base. O holdout não é usado até à secção 8. Os espaços de procura estão em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling.BASE_MODELS</code>.</p></div></div>

In [ ]:
# Procurar os melhores hiperparâmetros de cada modelo (validação cruzada com 5 folds)
# A rede neuronal pode avisar que não convergiu em algumas combinações: é esperado no tuning.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
best_params, tuning = modelling.tune_base_models(X_train, y_train, NUM, CAT, n_iter=N_ITER)

tuning.to_csv(TAB_DIR / "10_tuning_modelos_base.csv")
display(style_table(tuning.sort_values("cv_auc", ascending=False), precision=4, highlight="cv_auc"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Comparação em validação cruzada e ensembles</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>

<p>Os modelos afinados são combinados de duas formas, com cinco membros de famílias diferentes (linear, rede neuronal, bagging e boosting):</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Soft Voting</b>: média simples das probabilidades</li><li><b style='color:#0EE071'>Stacking</b>: um meta-modelo logístico aprende o peso de cada membro a partir de previsões <i>out-of-fold</i></li></ul>
<p>Todos os modelos, base e ensembles, são avaliados no <b style='color:#3777FF'>mesmo 5-fold</b> do treino com as nove métricas. Para cada modelo guardamos a média e o desvio-padrão em validação, a média no treino (para medir o sobreajuste) e as <b style='color:#3777FF'>previsões out-of-fold</b>: cada cliente recebe a probabilidade do modelo que não o viu. É com elas que se fazem os gráficos desta secção, sem tocar no holdout.</p>
<p>Esta é a célula mais lenta do notebook: o stacking treina os cinco membros dentro de cada fold.</p></div></div>

In [ ]:
# Linha de base + 7 modelos afinados + 2 ensembles
models = modelling.build_models(NUM, CAT, best_params)

# Todos no mesmo 5-fold: nove métricas e previsões out-of-fold
cv_table, oof = modelling.cv_evaluate(models, X_train, y_train)
cv_table.to_csv(TAB_DIR / "11_cv_modelos.csv")

cv_view = modelling.metrics_view(cv_table)            # médias em CV (modelos x métricas)
cv_train = modelling.metrics_view(cv_table, "train")  # médias no treino
cv_std = cv_table[[m + "_std" for m in modelling.METRICS]]
cv_std.columns = list(modelling.METRICS)
display(style_table(cv_view, precision=3, highlight="auc"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.1 · Diversidade entre modelos</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um ensemble só ganha se os membros errarem em clientes diferentes. A matriz mostra a correlação entre as probabilidades out-of-fold de cada par de modelos: valores perto de 1 indicam modelos que "pensam" igual; valores mais baixos indicam erros diferentes, que se compensam na média.</p></div></div>

In [ ]:
corr = modelling.prediction_correlation(oof)
corr.to_csv(TAB_DIR / "11c_correlacao_modelos.csv")
model_plots.plot_diversity(corr, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.2 · ROC AUC e seleção do modelo final</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O modelo final é o de <b style='color:#0EE071'>maior ROC AUC médio em CV</b>, escolhido antes de olhar para o holdout. A barra de erro é o desvio-padrão entre folds; a linha a tracejado é o AUC do Churn Score IBM no treino, a fasquia a bater.</p>
<p>Modelos a menos de um desvio-padrão do vencedor estão, na prática, <b style='color:#3777FF'>empatados</b>: a diferença está dentro do ruído entre folds. Nesse caso a escolha pode ser revista em favor do modelo mais simples ou mais bem calibrado (Brier mais baixo), e a decisão deve ficar justificada no relatório.</p></div></div>

In [ ]:
# AUC do Churn Score da IBM no treino (a fasquia a bater)
auc_ibm_train = modelling.score_metrics(y_train, train["Churn Score"] / 100)["auc"]
model_plots.plot_cv_comparison(cv_table, benchmark_auc=auc_ibm_train, show=SHOW)

# Escolher o modelo com maior AUC (a linha de base não é candidata)
BEST = modelling.select_best(cv_table)
empatados = modelling.tied_with_best(cv_table, BEST)

print("Modelo selecionado (maior AUC em CV):", BEST, "| AUC", round(cv_table.loc[BEST, "auc"], 4),
      "±", round(cv_table.loc[BEST, "auc_std"], 4))
print("Empatados com o selecionado (diferença < 1 desvio-padrão):", empatados or "nenhum")
print("Referência - AUC do Churn Score IBM no treino:", round(auc_ibm_train, 3))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.3 · As nove métricas, modelo a modelo</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Um painel por métrica e uma barra por modelo (média em CV, com o desvio-padrão). O <b style='color:#3777FF'>losango</b> é o valor no treino. Ensembles e modelo final a <b style='color:#0EE071'>verde</b>, modelos individuais a <b style='color:#3777FF'>azul</b>, linha de base a cinzento.</p>
<p>Como ler: a linha de base tem accuracy alta e precision/recall a zero, porque nunca sinaliza ninguém. Nas métricas com limiar 0,5, a maioria dos modelos tem precision acima do recall: com 28% de churn, o limiar 0,5 é exigente (ver 5.7).</p></div></div>

In [ ]:
model_plots.plot_metrics_panel(cv_view, std=cv_std, train=cv_train, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.4 · Grelha modelo × métrica</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O mesmo resumo numa grelha. A cor indica a <b style='color:#3777FF'>posição</b> do modelo em cada métrica (azul mais forte = melhor), já invertida no log loss e no Brier; o número é o valor real. Ajuda a ver se um modelo é bom em tudo ou só nalgumas métricas.</p></div></div>

In [ ]:
model_plots.plot_metrics_heatmap(cv_view, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.5 · Sobreajuste: treino vs validação</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>ROC AUC no treino e em validação cruzada. Quanto maior a distância (Δ), mais o modelo decorou o treino. As florestas e o boosting têm sempre algum Δ, porque cada árvore aprende bem o seu treino; o que importa é o valor em CV, que é o que se espera em clientes novos.</p></div></div>

In [ ]:
model_plots.plot_overfitting(cv_table.drop(index=modelling.BASELINE), show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.6 · Curvas ROC e Precision-Recall</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Curvas de todos os modelos com as previsões out-of-fold, mais o Churn Score IBM a tracejado preto. O modelo final está a <b style='color:#0EE071'>verde e mais grosso</b>.</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>ROC</b>: % de churners apanhados (vertical) contra % de clientes fiéis sinalizados por engano (horizontal), em todos os limiares</li><li><b style='color:#3777FF'>Precision-Recall</b>: dos clientes sinalizados, quantos saíram de facto, à medida que se apanham mais churners. A linha pontilhada é a taxa de churn, o que se consegue ao acaso</li></ul></div></div>

In [ ]:
oof_com_ibm = oof | {"Churn Score IBM": train["Churn Score"].to_numpy() / 100}
model_plots.plot_roc_pr(y_train, oof_com_ibm, best=BEST, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.7 · Matrizes de confusão</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Para cada modelo, no limiar 0,5: nº de clientes e % da classe real. A diagonal são os acertos. O quadrado de <b style='color:#3777FF'>baixo à esquerda</b> são os churners que o modelo não detetou, o erro mais caro para a retenção; o de cima à direita são clientes fiéis contactados sem necessidade.</p></div></div>

In [ ]:
model_plots.plot_confusion_matrices(y_train, oof, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.8 · Limiar de decisão do modelo final</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>O modelo dá uma probabilidade; para agir é preciso um limiar. Baixar o limiar apanha mais churners (recall sobe) mas contacta mais clientes que ficariam (precision desce). A linha a tracejado marca o limiar que maximiza o F1, como referência.</p>
<p>Na prática, a decisão faz-se com a lista ordenada por <b style='color:#0EE071'>P(churn) × CLTV</b> e um orçamento de contactos (secção 10); o limiar serve para traduzir esse orçamento numa regra e para discutir custos com o Head of Customer Retention.</p></div></div>

In [ ]:
limiar_f1 = modelling.best_f1_threshold(y_train, oof[BEST])
limiares = modelling.threshold_table(y_train, oof[BEST])
limiares.to_csv(TAB_DIR / "11b_limiares_modelo_final.csv")
display(style_table(limiares, precision=3))
model_plots.plot_threshold(y_train, oof[BEST], BEST, marker=limiar_f1, show=SHOW)
print("Limiar que maximiza o F1 (out-of-fold):", round(limiar_f1, 2))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>5.9 · Como treinou a rede neuronal</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>

<p>Treinamos a rede com os hiperparâmetros escolhidos e mostramos a evolução: a <b style='color:#3777FF'>log loss no treino</b> desce a cada época, e a <b style='color:#0EE071'>accuracy na validação interna</b> (15% do treino) sobe até estabilizar. A linha a tracejado é a melhor época; o treino pára 20 épocas depois, se não houver melhoria.</p></div></div>

In [ ]:
mlp_pipeline = models["Neural Network (MLP)"].fit(X_train, y_train)
mlp = mlp_pipeline.named_steps["clf"]
print("Arquitetura:", mlp.hidden_layer_sizes, "| ativação:", mlp.activation,
      "| épocas treinadas:", mlp.n_iter_)
model_plots.plot_mlp_training(mlp, show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · Satisfaction Score — com e sem</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>No notebook 01, <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> 1–2 corresponde a 100% de churn: suspeita de ser registada no contexto da saída.
Treinamos o modelo selecionado <b style='color:#0EE071'>com e sem</b> a variável, no mesmo 5-fold, e documentamos a diferença.</p>
<p>Se o ganho for grande, é sinal de leakage: a versão com a variável <b style='color:#3777FF'>não serve para retenção preventiva</b>, porque o score só existe tarde demais.</p></div></div>

In [ ]:
# Modelo A: SEM Satisfaction Score (é o modelo escolhido na secção 5; o CV já foi feito)
cv_sem = cv_table.loc[[BEST], ["auc", "auc_std", "pr_auc", "brier"]]
cv_sem.index = [BEST + " — sem Satisfaction"]

# Modelo B: COM Satisfaction Score (igual, mas com mais uma coluna numérica)
NUM_SAT = NUM + ["Satisfaction Score"]
FEATURES_SAT = NUM_SAT + CAT
modelo_com = modelling.build_models(NUM_SAT, CAT, best_params)[BEST]
cv_com = modelling.cv_compare({BEST + " — com Satisfaction": modelo_com}, train[FEATURES_SAT], y_train)

# Juntar as duas linhas numa só tabela
sat_cv = pd.concat([cv_sem, cv_com])
display(style_table(sat_cv, precision=4, highlight="auc"))

auc_sem = cv_sem["auc"].iloc[0]   # iloc[0] = primeira (e única) linha
auc_com = cv_com["auc"].iloc[0]
print("Diferença de AUC (com - sem):", round(auc_com - auc_sem, 4))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>7 · City com TargetEncoder</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> tem ~1.100 valores distintos: one-hot é inviável. O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TargetEncoder</code> substitui cada cidade pela taxa de churn suavizada,
calculada com <i>cross-fitting</i> e <b style='color:#0EE071'>dentro do Pipeline</b> — em cada fold só vê o treino desse fold.</p>
<p>Só entra no modelo final se melhorar o AUC em CV de forma clara (mais do que o desvio-padrão entre folds).</p></div></div>

In [ ]:
# Mesmo modelo, mas com a coluna City (codificada com TargetEncoder)
FEATURES_CITY = FEATURES + ["City"]
city_model = modelling.build_models(NUM, CAT, best_params, te=["City"])[BEST]
city_cv = modelling.cv_compare({BEST + " + City (TargetEncoder)": city_model}, train[FEATURES_CITY], y_train)

variants = pd.concat([sat_cv, city_cv])
variants.to_csv(TAB_DIR / "12_variantes_satisfaction_city.csv")
display(style_table(variants, precision=4, highlight="auc"))

# Só usamos City se o ganho for maior do que o desvio-padrão entre folds
auc_com_city = city_cv["auc"].iloc[0]
auc_sem_city = cv_table.loc[BEST, "auc"]
desvio = cv_table.loc[BEST, "auc_std"]

gain = auc_com_city - auc_sem_city
USE_CITY = gain > desvio

print("Ganho de AUC com City:", round(gain, 4))
print("Incluir City no modelo final?", USE_CITY)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>8 · Avaliação final no holdout</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O holdout (20%) é usado <b style='color:#0EE071'>uma única vez</b>: todos os modelos são treinados no treino completo e avaliados no teste.
O modelo final já foi escolhido na secção 5; os restantes aparecem só para transparência.</p>
<p>O <b style='color:#3777FF'>Churn Score IBM</b> é avaliado no mesmo holdout — é a fasquia a bater. O seu Brier é dado apenas como referência: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score / 100</code> não é uma probabilidade calibrada. A versão com Satisfaction Score é reportada à parte, para documentar o efeito do leakage. A tabela tem as nove métricas para todos os modelos, incluindo a rede neuronal.</p></div></div>

In [ ]:
# Definir o modelo final e as suas colunas
if USE_CITY:
    final_model = city_model
    FINAL_FEATURES = FEATURES_CITY
else:
    final_model = models[BEST]
    FINAL_FEATURES = FEATURES

# Treinar cada modelo no treino completo e prever P(churn) no teste
# predict_proba devolve 2 colunas: [P(fica), P(sai)] -> ficamos com a coluna 1
probas = {}
for name in models:
    model = models[name]
    model.fit(X_train, y_train)
    probas[name] = model.predict_proba(X_test)[:, 1]

# Se o modelo final usa City, treiná-lo com essas colunas e substituir a previsão
if USE_CITY:
    final_model.fit(train[FINAL_FEATURES], y_train)
    probas[BEST] = final_model.predict_proba(test[FINAL_FEATURES])[:, 1]

# Versão com Satisfaction Score (só para mostrar o efeito do leakage)
modelo_com.fit(train[FEATURES_SAT], y_train)
probas[BEST + " + Satisfaction (leakage)"] = modelo_com.predict_proba(test[FEATURES_SAT])[:, 1]

# Tabela de métricas no teste (inclui o Churn Score IBM)
holdout = modelling.holdout_report(probas, y_test, test["Churn Score"])
holdout.to_csv(TAB_DIR / "13_holdout.csv")
display(style_table(holdout, precision=4, highlight="auc"))

# Curva ROC e calibração
p_test = probas[BEST]   # P(churn) do modelo final para cada cliente do teste
curvas = {
    BEST: p_test,
    "Churn Score IBM": test["Churn Score"] / 100,
    "Logistic Regression": probas["Logistic Regression"],
    "Neural Network (MLP)": probas["Neural Network (MLP)"],
}
model_plots.plot_roc_calibration(y_test, curvas, show=SHOW)

# As nove métricas no holdout, modelo a modelo (só para transparência: a escolha já foi feita)
model_plots.plot_metrics_panel(holdout.drop(index=modelling.BASELINE), best=BEST,
                               name="24b_painel_metricas_holdout",
                               title="Nove métricas por modelo — holdout (20%)", show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>9 · Interpretação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><b style='color:#0EE071'>Permutation importance</b> no holdout: quanto cai o AUC quando os valores de uma variável são baralhados.
Funciona para qualquer modelo (incluindo ensembles) e ao nível das variáveis originais, não das colunas one-hot — mais útil para explicar ao Head of Customer Retention.</p>
<p>Barras <b style='color:#3777FF'>azuis</b> = numéricas, <b style='color:#0EE071'>verdes</b> = categóricas.</p></div></div>

In [ ]:
# Quanto cai o AUC quando baralhamos os valores de cada coluna?
pi = permutation_importance(final_model, test[FINAL_FEATURES], y_test, scoring="roc_auc",
                            n_repeats=5, random_state=0, n_jobs=-1)

importance = pd.Series(pi.importances_mean, index=FINAL_FEATURES)
importance = importance.clip(lower=0)                  # valores negativos passam a 0
importance = importance.sort_values(ascending=False)   # da mais para a menos importante
importance.to_frame("queda_auc").to_csv(TAB_DIR / "14_importancia_permutacao.csv")

titulo = "Permutation importance — " + BEST + " (queda de AUC, holdout)"
plots.plot_ranking(importance.head(20), titulo, "27_importancia_permutacao",
                   highlight=set(CAT + ["City"]), show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>10 · Priorização: P(churn) × CLTV</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O objetivo não é só prever churn, é decidir <b style='color:#3777FF'>quem contactar primeiro</b>. Cada cliente recebe uma
<b style='color:#0EE071'>perda esperada = P(churn) × CLTV</b> e a lista de intervenção é ordenada por esse valor.</p>
<p>A <b style='color:#0EE071'>curva de captura</b> mede, no holdout, que % do CLTV dos clientes que de facto saíram fica coberta ao contactar os top-k% da lista.
Compara quatro ordenações: o nosso P × CLTV, só P(churn), o Churn Score IBM × CLTV e o Churn Score IBM sozinho.</p></div></div>

In [ ]:
# 4 formas diferentes de ordenar os clientes do teste
orderings = {
    BEST + ": P × CLTV": p_test * test["CLTV"],
    BEST + ": só P(churn)": p_test,
    "IBM: Churn Score × CLTV": test["Churn Score"] * test["CLTV"],
    "IBM: só Churn Score": test["Churn Score"],
}

# Que % do CLTV perdido apanhamos ao contactar os primeiros k% de cada lista?
curve = modelling.capture_curve(y_test, test["CLTV"], orderings)
capture = modelling.capture_at(curve, cuts=(0.1, 0.2, 0.3))
capture.to_csv(TAB_DIR / "15_captura_valor.csv")
display(style_table(capture, precision=3))
model_plots.plot_capture_curve(curve, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.1 · Lista de intervenção</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Lista ordenada para o holdout (onde podemos verificar o resultado) e para os clientes <b style='color:#3777FF'>Joined</b>, que ficaram fora do treino.
O mapa mostra onde ficam os top-N: o P × CLTV troca alguns clientes de risco alto mas pouco valor por clientes de valor alto com risco moderado.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TOP_N</code> é o orçamento de contactos — ajustar ao que a equipa de retenção consegue tratar.</p>
<p><b style='color:#0EE071'>Atenção aos Joined:</b> são clientes com poucos meses de casa. Na população de treino, tenure baixa está fortemente associada a churn,
por isso o modelo tende a dar-lhes P(churn) muito alta — é <b style='color:#3777FF'>extrapolação</b>, não um resultado validado.
A lista dos Joined deve ser lida como ordenação relativa e não como probabilidade calibrada.</p></div></div>

In [ ]:
TOP_N = 200   # nº de clientes que a equipa de retenção consegue contactar

# Lista de prioridade do teste + coluna com o churn real (para verificar)
prio_test = modelling.priority_list(test, p_test)
churn_real = test[["Customer ID", TARGET]]
prio_test = prio_test.merge(churn_real, on="Customer ID")
model_plots.plot_priority_map(prio_test, TOP_N, show=SHOW)

# Resumo por decil (decil 1 = 10% de clientes com maior perda esperada)
grupos = prio_test.groupby("decil")
by_decile = pd.DataFrame({
    "clientes": grupos.size(),
    "p_churn_media": grupos["p_churn"].mean(),
    "churn_real": grupos[TARGET].mean(),
    "cltv_medio": grupos["CLTV"].mean(),
    "perda_esperada": grupos["perda_esperada"].sum(),
})
display(style_table(by_decile, precision=3, highlight="churn_real"))
display(style_table(prio_test.head(15), precision=3))

In [ ]:
# Clientes Joined: novos clientes, sem resposta conhecida -> só prever (é extrapolação, ver nota acima)
p_joined = final_model.predict_proba(joined[FINAL_FEATURES])[:, 1]
prio_joined = modelling.priority_list(joined, p_joined)

print("Joined:", len(prio_joined), "clientes | P(churn) média:", round(p_joined.mean() * 100, 1), "%")
display(style_table(prio_joined.head(15), precision=3))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>11 · Exportação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Ficheiros gerados (ignorados pelo Git):</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/final_model.joblib</code> — modelo final treinado, lista de features e métricas</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/prioridade_holdout.csv</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>prioridade_joined.csv</code> — listas de intervenção ordenadas</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/tables/10_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>15_*</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/figures/20_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>27_*</code> — tabelas e gráficos desta fase (lista completa em docs/modelling.md)</li></ul></div></div>

In [ ]:
# Guardar o modelo final e a informação necessária para o voltar a usar
bundle = {
    "model": final_model,
    "name": BEST,
    "features": FINAL_FEATURES,
    "target": TARGET,
    "best_params": best_params,
    "cv": cv_table.loc[BEST].to_dict(),
    "holdout": holdout.loc[BEST].to_dict(),
}
joblib.dump(bundle, ARTIFACTS_DIR / "final_model.joblib")

# Guardar as listas de prioridade
prio_test.to_csv(DATA_PROCESSED / "prioridade_holdout.csv", index=False)
prio_joined.to_csv(DATA_PROCESSED / "prioridade_joined.csv", index=False)

print("Guardado: outputs/artifacts/final_model.joblib")
print("Guardado: data/processed/prioridade_holdout.csv")
print("Guardado: data/processed/prioridade_joined.csv")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>12 · Conclusões para o relatório</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>A célula seguinte resume os números-chave a partir das tabelas acima, para transcrever para o relatório:</p>
<ul style='margin:6px 0 2px 0'><li>modelos empatados com o vencedor em CV e desempenho da rede neuronal</li><li>desempenho do modelo final vs <b style='color:#3777FF'>Churn Score IBM</b> no mesmo holdout</li><li>efeito do <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> (leakage) e decisão sobre <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code></li><li>ganho de valor captado com a priorização <b style='color:#0EE071'>P(churn) × CLTV</b></li></ul></div></div>

In [ ]:
# Números-chave para o relatório
auc_modelo = holdout.loc[BEST, "auc"]
auc_ibm = holdout.loc["Churn Score IBM (benchmark)", "auc"]
pr_auc = holdout.loc[BEST, "pr_auc"]
brier = holdout.loc[BEST, "brier"]
auc_sat = holdout.loc[BEST + " + Satisfaction (leakage)", "auc"]

captura_modelo = capture.loc[BEST + ": P × CLTV", "top 20%"]
captura_ibm = capture.loc["IBM: só Churn Score", "top 20%"]

print("Modelo final:", BEST)
print("  Empatados em CV (diferença < 1 desvio-padrão):", empatados or "nenhum")
print("  Rede neuronal (MLP): AUC CV", round(cv_table.loc["Neural Network (MLP)", "auc"], 3),
      "| AUC teste", round(holdout.loc["Neural Network (MLP)", "auc"], 3))
print("  AUC no teste:", round(auc_modelo, 3))
print("  AUC do IBM:  ", round(auc_ibm, 3))
print("  Diferença:   ", round(auc_modelo - auc_ibm, 3))
print("  PR-AUC:", round(pr_auc, 3), "| Brier:", round(brier, 3))
print("Satisfaction Score: AUC com a variável =", round(auc_sat, 3), "(fora do modelo final)")
print("City incluída no modelo final?", USE_CITY, "(ganho CV:", round(gain, 4), ")")
print("Contactando 20% dos clientes:")
print("  P × CLTV capta", round(captura_modelo * 100, 1), "% do CLTV perdido")
print("  Churn Score IBM capta", round(captura_ibm * 100, 1), "%")